In [1]:
import sys , os
import pandas as pd
import re

sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

# Part 01

In [ ]:
file_path = '../data/Sample Messages.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()


cleaned_text = re.sub(r'\\s*', '', raw_text)
messages_list = [line.strip() for line in cleaned_text.split('\n') if line.strip()]

print(f"Total messages to process: {len(messages_list)}\n")


model = pick_model('google', 'general')
client = LLMClient('google', model)


results = []

examples = """
Example 1:
Message: "Breaking News: Kelani River level at 9m."
Output: District: Colombo | Intent: Info | Priority: Low

Example 2:
Message: "We are trapped on the roof with 3 kids!"
Output: District: None | Intent: Rescue | Priority: High

Example 3:
Message: "Does anyone have extra dry rations for the camp in Gampaha?"
Output: District: Gampaha | Intent: Supply | Priority: High

Example 4:
Message: "Just arrived in Galle. Weather is nice."
Output: District: Galle | Intent: Other | Priority: Low
"""


for msg in messages_list[:4]:
    

    prompt_text, spec = render(
        'few_shot.v1',
        role='rescue, supply, info, other classifier',
        examples=examples,
        query=f'Message: "{msg}"',
        constraints='Follow the pattern in examples: provide output in the format "District: <district> | Intent: <intent> | Priority: <priority>". If the district is not mentioned, use "None".',
        format='District: {{district}} | Intent: {{intent}} | Priority: {{priority}}'
    )

    chat_messages = [{'role': 'user', 'content': prompt_text}]
    

    response = client.chat(chat_messages, temperature=0.1) 
    

    output_text = response['text'].strip()
    

    results.append({
        'Original Message': msg,
        'Classification': output_text
    })
    
    print(f"Processed: {msg[:40]}... -> {output_text}")
    

    if 'latency_ms' in response:
        log_llm_call('gemini', model, 'few_shot', response['latency_ms'], response.get('usage', {}))


os.makedirs('../output', exist_ok=True)
df = pd.DataFrame(results)


output_file = '../output/classified_messages.xlsx'
df.to_excel(output_file, index=False)

print(f"\nDone! All results saved to {output_file}")

# Part 02

In [ ]:
import re
from IPython.display import Markdown, display

print("--- Starting Part 2: Stability Experiment ---\n")


file_path = '../data/Scenarios.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_scenarios = f.read()


cleaned_scenarios = re.sub(r'\\s*', '', raw_scenarios)


scenarios_list = [s.strip() for s in cleaned_scenarios.split('SCENARIO') if s.strip()]
scenarios_list = ['SCENARIO ' + s for s in scenarios_list]

print(f"Found {len(scenarios_list)} scenarios to test.\n")


reasoning_model = pick_model('google', 'cot') 
print(f'Using reasoning model: {reasoning_model}\n')
client_reasoning = LLMClient('google', reasoning_model)


for scenario in scenarios_list:
    print("="*80)
    print(f"Testing Scenario:\n{scenario[:150]}...\n") 
    print("="*80)
    

    prompt_text, spec = render(
        'cot_reasoning.v1', 
        role='expert emergency response coordinator',
        problem=scenario
    )
    
    chat_messages = [{'role': 'user', 'content': prompt_text}]
    
    # ---------------------------------------------------------
    # SAFE MODE (Temperature 0.0) 
    # ---------------------------------------------------------
    print("\n🟢 [ SAFE MODE | Temperature = 0.0 ]\n")
    response_safe = client_reasoning.chat(
        chat_messages, 
        temperature=0.0, 
        max_tokens=spec.max_tokens
    )
    display(Markdown(response_safe['text']))
    
    # ---------------------------------------------------------
    # CHAOS MODE (Temperature 1.0) - Running 3 times
    # ---------------------------------------------------------
    print("\n\n🔴 [ CHAOS MODE | Temperature = 1.0 ] (Running 3 times)")
    print("-" * 60)
    
    for i in range(1, 4):
        print(f"\n--- Chaos Run {i} ---")
        response_chaos = client_reasoning.chat(
            chat_messages, 
            temperature=1.0, 
            max_tokens=spec.max_tokens
        )
        display(Markdown(response_chaos['text']))
        
    print("\n" + "#"*80 + "\n")

# Part 03

In [ ]:
import os
import re
from IPython.display import Markdown, display


print("--- Starting Part 3: The Logistics Commander (CoT & ToT) ---\n")


file_path = '../data/Incidents.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_incidents = f.read()


cleaned_incidents = re.sub(r'\\s*', '', raw_incidents).strip()
print("Data Loaded Successfully!\n")
print("-" * 60)

# ==========================================
# STEP A: CoT - Scoring 
# ==========================================
print("Executing Step A (CoT - Scoring)...")

reasoning_model = pick_model('google', 'cot')
client_cot = LLMClient('google', reasoning_model)


step_a_logic = f"""
Analyze the following 3 incidents row by row and assign a Priority Score (1-10) to each.
Logic:
- Base Score: 5
- +2 if Age > 60 or Age < 5
- +3 if Need == "Rescue" (Life Threat)
- +1 if Need == "Insulin/Medicine"

Incidents Data:
{cleaned_incidents}
"""

prompt_a, spec_a = render(
    'cot_reasoning.v1', 
    role='expert emergency response coordinator',
    problem=step_a_logic
)

response_a = client_cot.chat([{'role': 'user', 'content': prompt_a}], temperature=0.0, max_tokens=spec_a.max_tokens)


display(Markdown("### Step A: CoT - Scoring Results"))
display(Markdown(response_a['text']))
print("-" * 60)

# ==========================================
# STEP B: ToT - Strategy 
# ==========================================
print("Executing Step B (ToT - Strategy)...")


strategy_model = pick_model('google', 'tot')
client_tot = LLMClient('google', strategy_model)

step_b_logic = f"""
Based on the scores calculated in Step A, determine the optimal rescue route.
Situation: You have ONE rescue boat starting at Ragama.

Original Incidents Data (Locations):
{cleaned_incidents}

Scores calculated in Step A:
{response_a['text']}

Constraints - Travel times:
- Ragama -> Ja-Ela: 10 mins
- Ja-Ela -> Gampaha: 40 mins

Task: Explore 3 distinct branches:
- Branch 1: Save the highest score first (Greedy).
- Branch 2: Save closest first (Speed).
- Branch 3: Save furthest first (Logistics).

Goal: Maximize total priority score saved within the shortest time. Assume the boat can handle one incident per stop.
Select the optimal route and explain why.

CRITICAL INSTRUCTION: Keep your reasoning STRICTLY CONCISE. Do not write long paragraphs. Summarize each branch in a maximum of 2 to 3 bullet points. Just output the math and the final decision.
"""

prompt_b, spec_b = render(
    'tot_reasoning.v1', 
    role='expert logistics commander',
    problem=step_b_logic,
    branches='3'
)

response_b = client_tot.chat([{'role': 'user', 'content': prompt_b}], temperature=0.0, max_tokens=spec_b.max_tokens)


display(Markdown("### Step B: ToT - Strategy Results"))
display(Markdown(response_b['text']))

print("\n--- Part 3 Execution Done! ---")

--- Starting Part 3: The Logistics Commander (CoT & ToT) ---

Data Loaded Successfully!

------------------------------------------------------------
Executing Step A (CoT - Scoring)...


### Step A: CoT - Scoring Results

Reasoning Steps:
1.  Initialize each incident's Priority Score with a Base Score of 5.
2.  For each incident, check the 'Ages' column: add 2 to the score if any age is greater than 60 or less than 5.
3.  For each incident, check the 'Main Need' column: add 3 to the score if it is "Rescue".
4.  For each incident, check the 'Main Need' column: add 1 to the score if it is "Insulin" or "Medicine".
5.  Sum all applicable points for each incident to get the final Priority Score.

Answer:
**Incident 1:**
*   Base Score: 5
*   Age (20-40): No age > 60 or < 5. (+0)
*   Need (Water): Not "Rescue". (+0)
*   Need (Water): Not "Insulin/Medicine". (+0)
*   **Priority Score: 5**

**Incident 2:**
*   Base Score: 5
*   Age (75): Age > 60. (+2)
*   Need (Insulin): Not "Rescue". (+0)
*   Need (Insulin): Is "Insulin/Medicine". (+1)
*   **Priority Score: 8**

**Incident 3:**
*   Base Score: 5
*   Age (10, 35): No age > 60 or < 5. (+0)
*   Need (Rescue): Is "Rescue". (+3)
*   Need (Rescue): Not "Insulin/Medicine". (+0)
*   **Priority Score: 8**

------------------------------------------------------------
Executing Step B (ToT - Strategy)...


### Step B: ToT - Strategy Results

Here are the explorations of the three distinct solution paths:

---
### Branch 1: Save the highest score first (Greedy)

*   **Hypothesis:** Prioritizing the most critical incidents first will yield the highest overall score, even if it takes longer.
*   **Steps:**
    1.  Start at Ragama. Highest scores are Incident 2 (Ja-Ela, 8) and Incident 3 (Ragama, 8). Choose Incident 3 due to 0 min travel time.
    2.  Ragama (Start) -> Incident 3 (Ragama): Time 0 min, Score 8. Current location: Ragama.
    3.  Next highest score is Incident 2 (Ja-Ela, 8). Travel Ragama -> Ja-Ela: Time 10 min, Score 8. Current location: Ja-Ela.
    4.  Last incident is Incident 1 (Gampaha, 5). Travel Ja-Ela -> Gampaha: Time 40 min, Score 5. Current location: Gampaha.
*   **Intermediate check:** Total Score: 8 + 8 + 5 = 21. Total Time: 0 + 10 + 40 = 50 min.
    *   Route: Ragama (Start) -> Incident 3 (Ragama) -> Incident 2 (Ja-Ela) -> Incident 1 (Gampaha)

---
### Branch 2: Save closest first (Speed)

*   **Hypothesis:** Minimizing travel time between incidents will allow more rescues in a shorter period.
*   **Steps:**
    1.  Start at Ragama. Closest incident is Incident 3 (Ragama, 0 min).
    2.  Ragama (Start) -> Incident 3 (Ragama): Time 0 min, Score 8. Current location: Ragama.
    3.  Next closest from Ragama is Incident 2 (Ja-Ela, 10 min). Travel Ragama -> Ja-Ela: Time 10 min, Score 8. Current location: Ja-Ela.
    4.  Last incident is Incident 1 (Gampaha, 40 min from Ja-Ela). Travel Ja-Ela -> Gampaha: Time 40 min, Score 5. Current location: Gampaha.
*   **Intermediate check:** Total Score: 8 + 8 + 5 = 21. Total Time: 0 + 10 + 40 = 50 min.
    *   Route: Ragama (Start) -> Incident 3 (Ragama) -> Incident 2 (Ja-Ela) -> Incident 1 (Gampaha)

---
### Branch 3: Save furthest first (Logistics)

*   **Hypothesis:** Addressing the furthest incident first might optimize the return journey or subsequent stops, preventing backtracking.
*   **Steps:**
    1.  Start at Ragama. Furthest incident is Incident 1 (Gampaha, 50 min via Ja-Ela).
    2.  Ragama (Start) -> Incident 1 (Gampaha): Time 50 min, Score 5. Current location: Gampaha.
    3.  Next furthest from Gampaha is Incident 3 (Ragama, 50 min via Ja-Ela). Travel Gampaha -> Ragama: Time 50 min, Score 8. Current location: Ragama.
    4.  Last incident is Incident 2 (Ja-Ela, 10 min from Ragama). Travel Ragama -> Ja-Ela: Time 10 min, Score 8. Current location: Ja-Ela.
*   **Intermediate check:** Total Score: 5 + 8 + 8 = 21. Total Time: 50 + 50 + 10 = 110 min.
    *   Route: Ragama (Start) -> Incident 1 (Gampaha) -> Incident 3 (Ragama) -> Incident 2 (Ja-Ela)

---
### Answer:

**Optimal Route:** Ragama (Start) -> Incident 3 (Ragama) -> Incident 2 (Ja-Ela) -> Incident 1 (Gampaha)

**Explanation:**
*   All three branches result in the same total priority score of 21, as all incidents are eventually addressed.
*   The goal is to maximize the total priority score within the shortest time.
*   Branch 1 (Greedy) and Branch 2 (Speed) both achieve the maximum score in the shortest time of 50 minutes.
*   Branch 3 (Logistics) takes significantly longer (110 minutes) to achieve the same score.
*   Therefore, the route starting with Incident 3 at Ragama, then proceeding to Ja-Ela, and finally Gampaha, is optimal for both score and time efficiency.


--- Part 3 Execution Done! ---
